In [1]:
import pandas as pd
skaters = pd.read_csv("../data/processed/skater_game_stats.csv")
goalies = pd.read_csv("../data/processed/goalie_game_stats.csv")
display(skaters.head())
display(goalies.head())

,game_id,team,is_home,player_id,player_name,position,goals,assists,points,plus_minus,pim,hits,pp_goals,shots,blocked_shots,giveaways,takeaways,shifts,faceoff_pct,toi_minutes
0,2023020001,TBL,True,8476822,L. Glendening,C,0,0,0,0,0,1,0,1,2,0,0,23,0.692308,14.016667
1,2023020001,TBL,True,8477426,N. Paul,L,2,1,3,1,0,2,2,3,1,2,1,19,0.272727,16.200000
2,2023020001,TBL,True,8478010,B. Point,C,0,3,3,1,0,0,0,3,1,0,0,24,0.000000,22.216667
3,2023020001,TBL,True,8479591,M. Eyssimont,C,0,0,0,0,0,0,0,2,0,0,2,14,0.000000,8.516667
4,2023020001,TBL,True,8479542,B. Hagel,L,1,0,1,0,0,0,0,4,1,0,0,20,1.000000,18.116667


,game_id,team,is_home,player_id,player_name,shots_against,saves,goals_against,save_pct,pp_goals_against,starter,decision,pim,toi_minutes
0,2023020001,TBL,True,8477992,J. Johansson,31,28,3,0.903226,1,True,W,0,60.000000
1,2023020001,TBL,True,8477035,M. Tomkins,0,0,0,NaN,0,False,NaN,0,0.000000
2,2023020001,NSH,False,8480947,K. Lankinen,0,0,0,NaN,0,False,NaN,0,0.000000
3,2023020001,NSH,False,8477424,J. Saros,33,29,4,0.878788,2,True,L,0,57.266667
4,2023020002,PIT,True,8477465,T. Jarry,35,32,3,0.914286,0,True,L,0,57.500000


In [2]:
skaters.groupby(["game_id", "team"]).size().value_counts().sort_index()

17      19
18    7853
Name: count, dtype: int64

In [5]:
goalies["played"] = goalies["toi_minutes"].fillna(0) > 0
print(goalies["played"].value_counts())
goalies.groupby(["game_id", "team"])["played"].sum().value_counts()

played
True     8329
False    7413
Name: count, dtype: int64


played
1    7415
2     457
Name: count, dtype: int64

In [4]:
games = pd.read_csv("../data/processed/games.csv")
outcomes = pd.read_csv("../data/processed/game_outcomes.csv")
team_goals = skaters.groupby(["game_id", "team"])["goals"].sum().reset_index()

g = games.merge(outcomes, on="game_id")
g = g[g["last_period_type"] == "REG"]
home = g.merge(team_goals, left_on=["game_id", "home_team"], right_on=["game_id", "team"])
(home["goals"] == home["home_score"]).mean()

np.float64(0.9996739484838605)

In [6]:
away = g.merge(team_goals, left_on=["game_id", "away_team"], right_on=["game_id", "team"])
print("away match rate:", (away["goals"] == away["away_score"]).mean())

cols = ["game_id", "game_date", "home_team", "away_team", "home_score", "away_score", "goals"]
display(home[home["goals"] != home["home_score"]][cols])
display(away[away["goals"] != away["away_score"]][cols])

away match rate: 0.9990218454515813


,game_id,game_date,home_team,away_team,home_score,away_score,goals
1856,2024020949,2025-03-01,NYI,NSH,7.0,4.0,6


,game_id,game_date,home_team,away_team,home_score,away_score,goals
947,2023020345,2023-11-30,TBL,PIT,2.0,4.0,3
1199,2024020720,2025-01-17,BUF,PIT,2.0,5.0,4
1732,2024020052,2024-10-15,STL,MIN,1.0,4.0,3


In [7]:
import json
d = json.load(open("../data/raw/boxscores/2024020949.json"))

for side in ["homeTeam", "awayTeam"]:
    groups = d["playerByGameStats"][side]
    total = sum(p.get("goals", 0) for players in groups.values() for p in players)
    print(side, d[side]["abbrev"], "| score:", d[side]["score"], "| goals in player stats:", total)
    for group, players in groups.items():
        for p in players:
            if p.get("goals", 0) > 0:
                print("   ", group, p["name"]["default"], p["goals"])

homeTeam NYI | score: 7 | goals in player stats: 6
    forwards B. Horvat 1
    forwards K. Palmieri 1
    forwards B. Nelson 2
    forwards J. Pageau 1
    forwards C. Cizikas 1
awayTeam NSH | score: 4 | goals in player stats: 4
    forwards C. Sissons 1
    forwards C. Smith 2
    defense M. Del Gaizo 1
